01. Calculo DAN con Reporte DANES

In [1]:
# ==================== PROCESAMIENTO COMPLETO DE ARCHIVOS DAN ====================
import pandas as pd
import os
import glob
from datetime import datetime

# Configuración de rutas
carpeta_dan = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Calculo de PC y DAN\LAM\DAN'
carpeta_outputs = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Calculo de PC y DAN\LAM\Outputs'

# Crear carpeta de outputs si no existe
os.makedirs(carpeta_outputs, exist_ok=True)

# PASO 1: EXPLORACIÓN DE LA CARPETA DAN
print("🔍 EXPLORANDO CARPETA DAN...")
print("="*60)

if os.path.exists(carpeta_dan):
    print(f"Contenido de la carpeta: {carpeta_dan}")
    print("-" * 50)
    
    # Listar todos los archivos en la carpeta
    archivos = os.listdir(carpeta_dan)
    
    if archivos:
        for i, archivo in enumerate(archivos, 1):
            ruta_completa = os.path.join(carpeta_dan, archivo)
            tamaño = os.path.getsize(ruta_completa) / 1024  # Tamaño en KB
            print(f"{i}. {archivo} ({tamaño:.1f} KB)")
            
        print(f"\nTotal de archivos encontrados: {len(archivos)}")
        
        # Buscar archivos Excel y CSV específicamente
        archivos_excel = glob.glob(os.path.join(carpeta_dan, '*.xlsx')) + glob.glob(os.path.join(carpeta_dan, '*.xlsm')) + glob.glob(os.path.join(carpeta_dan, '*.xls'))
        archivos_csv = glob.glob(os.path.join(carpeta_dan, '*.csv'))
        
        if archivos_excel:
            print(f"\nArchivos Excel encontrados: {len(archivos_excel)}")
            for excel_file in archivos_excel:
                print(f"- {os.path.basename(excel_file)}")
        
        if archivos_csv:
            print(f"\nArchivos CSV encontrados: {len(archivos_csv)}")
            for csv_file in archivos_csv:
                print(f"- {os.path.basename(csv_file)}")
                
        if not archivos_excel and not archivos_csv:
            print("\nNo se encontraron archivos Excel ni CSV en la carpeta.")
            
    else:
        print("La carpeta está vacía.")
        
else:
    print(f"La carpeta no existe: {carpeta_dan}")

print("\n" + "="*60)

# PASO 2: PROCESAMIENTO AUTOMÁTICO DE ARCHIVOS
print("\n🔍 PROCESANDO ARCHIVOS DAN (Excel y CSV)...")
print("="*60)

# Inicializar lista para almacenar todos los DataFrames
list_df_danes = []

# Buscar todos los archivos Excel y CSV en la carpeta DAN
archivos_excel = glob.glob(os.path.join(carpeta_dan, '*.xlsx')) + \
                 glob.glob(os.path.join(carpeta_dan, '*.xlsm')) + \
                 glob.glob(os.path.join(carpeta_dan, '*.xls'))
archivos_csv = glob.glob(os.path.join(carpeta_dan, '*.csv'))

# Combinar todos los archivos
todos_archivos = archivos_excel + archivos_csv

if todos_archivos:
    for archivo in todos_archivos:
        nombre_archivo = os.path.basename(archivo)
        extension = os.path.splitext(archivo)[1].lower()
        print(f"\n📂 Procesando: {nombre_archivo} (Tipo: {extension})")
        
        try:
            if extension == '.csv':
                # Procesar archivo CSV
                try:
                    # Intentar leer con diferentes encodings
                    encodings = ['utf-8', 'latin-1', 'cp1252', 'iso-8859-1']
                    df_temp = None
                    
                    for encoding in encodings:
                        try:
                            df_temp = pd.read_csv(archivo, encoding=encoding)
                            print(f"   ✅ CSV leído exitosamente con encoding: {encoding}")
                            break
                        except UnicodeDecodeError:
                            continue
                    
                    if df_temp is None:
                        print(f"   ❌ No se pudo leer el CSV con ningún encoding probado")
                        continue
                    
                    if not df_temp.empty:
                        # Agregar columnas de metadatos para identificar origen
                        df_temp['ARCHIVO_ORIGEN'] = nombre_archivo
                        df_temp['HOJA_ORIGEN'] = 'CSV_DATA'
                        
                        # Estandarizar nombres de columnas
                        df_temp.columns = df_temp.columns.str.upper().str.strip()
                        
                        # Agregar a la lista
                        list_df_danes.append(df_temp)
                        print(f"   ✅ CSV procesado: {df_temp.shape[0]} filas")
                    else:
                        print(f"   ⚠️ CSV vacío, omitido")
                        
                except Exception as e:
                    print(f"   ❌ Error procesando CSV: {e}")
                    
            else:
                # Procesar archivo Excel
                excel_file = pd.ExcelFile(archivo)
                print(f"   Hojas encontradas: {len(excel_file.sheet_names)} -> {excel_file.sheet_names}")
                
                # Procesar cada hoja del archivo
                for hoja in excel_file.sheet_names:
                    try:
                        df_temp = pd.read_excel(archivo, sheet_name=hoja)
                        
                        if not df_temp.empty:
                            # Agregar columnas de metadatos para identificar origen
                            df_temp['ARCHIVO_ORIGEN'] = nombre_archivo
                            df_temp['HOJA_ORIGEN'] = hoja
                            
                            # Estandarizar nombres de columnas
                            df_temp.columns = df_temp.columns.str.upper().str.strip()
                            
                            # Agregar a la lista
                            list_df_danes.append(df_temp)
                            print(f"   ✅ Hoja '{hoja}': {df_temp.shape[0]} filas procesadas")
                        else:
                            print(f"   ⚠️ Hoja '{hoja}': vacía, omitida")
                            
                    except Exception as e:
                        print(f"   ❌ Error procesando hoja '{hoja}': {e}")
                        
        except Exception as e:
            print(f"❌ Error procesando archivo '{nombre_archivo}': {e}")

    # PASO 3: CONSOLIDACIÓN Y TRANSFORMACIÓN DE DATOS
    if list_df_danes:
        df_danes = pd.concat(list_df_danes, ignore_index=True)
        
        # Crear la columna CONCA concatenando RUTA y CLAVEPRODUCTO
        if 'CLAVEDOCUMENTO' in df_danes.columns and 'CLAVEPRODUCTO' in df_danes.columns:
            import re
            
            def extraer_ruta(clavedocumento):
                if pd.isna(clavedocumento):
                    return None
                # Extraer solo los números de CLAVEDOCUMENTO para crear RUTA
                numeros = re.findall(r'\d+', str(clavedocumento))
                if numeros:
                    # Tomar el primer grupo de números encontrado
                    return numeros[0]
                return None
            
            def crear_conca(row):
                ruta = extraer_ruta(row['CLAVEDOCUMENTO'])
                claveproducto = row['CLAVEPRODUCTO']
                
                if pd.isna(ruta) or pd.isna(claveproducto):
                    return None
                # Concatenar RUTA + CLAVEPRODUCTO
                return f"{ruta}{claveproducto}"
            
            # Crear columna RUTA
            df_danes['RUTA'] = df_danes['CLAVEDOCUMENTO'].apply(extraer_ruta)
            
            # Crear columna CONCA
            df_danes['CONCA'] = df_danes.apply(crear_conca, axis=1)
            
            print(f"\n✅ Columnas RUTA y CONCA creadas exitosamente")
            
            # Mostrar algunos ejemplos de la transformación
            ejemplos = df_danes[['CLAVEDOCUMENTO', 'RUTA', 'CLAVEPRODUCTO', 'CONCA']].dropna().head(3)
            if not ejemplos.empty:
                print("📝 Ejemplos de transformación:")
                for _, row in ejemplos.iterrows():
                    print(f"   {row['CLAVEDOCUMENTO']} → RUTA: {row['RUTA']} + PRODUCTO: {row['CLAVEPRODUCTO']} = CONCA: {row['CONCA']}")
        else:
            print("⚠️ No se encontraron las columnas CLAVEDOCUMENTO o CLAVEPRODUCTO para crear CONCA")
        
        # PASO 4: AGRUPACIÓN POR CONCA
        if 'CONCA' in df_danes.columns and 'CANTIDAD' in df_danes.columns:
            print(f"\n📊 AGRUPANDO POR CONCA...")
            print("-" * 40)
            
            # Crear DataFrame agrupado
            df_danes_agrupado = df_danes.groupby('CONCA').agg({
                'CANTIDAD': 'sum',
                'CLAVEDOCUMENTO': 'first',
                'RUTA': 'first',
                'CLAVEPRODUCTO': 'first'
            }).reset_index()
            
            print(f"✅ Agrupación completada:")
            print(f"   Registros originales: {df_danes.shape[0]}")
            print(f"   Registros agrupados: {df_danes_agrupado.shape[0]}")
            print(f"   Reducción: {df_danes.shape[0] - df_danes_agrupado.shape[0]} registros consolidados")
            
            # Mostrar primeros resultados agrupados
            print(f"\n📝 Primeros 5 registros agrupados:")
            print(df_danes_agrupado.head().to_string())
            
            # Mostrar estadísticas de cantidad
            print(f"\n📈 Estadísticas de CANTIDAD agrupada:")
            print(f"   Total cantidad: {df_danes_agrupado['CANTIDAD'].sum():,.0f}")
            print(f"   Cantidad promedio: {df_danes_agrupado['CANTIDAD'].mean():.2f}")
            print(f"   Cantidad máxima: {df_danes_agrupado['CANTIDAD'].max():,.0f}")
            print(f"   Cantidad mínima: {df_danes_agrupado['CANTIDAD'].min():,.0f}")
            
        else:
            print("⚠️ No se pueden agrupar los datos: faltan columnas CONCA o CANTIDAD")
            df_danes_agrupado = None
        
        # PASO 5: RESUMEN DE RESULTADOS
        print(f"\n📊 RESULTADO CONSOLIDADO:")
        print("="*60)
        print(f"Total de registros: {df_danes.shape[0]}")
        print(f"Total de columnas: {df_danes.shape[1]}")
        
        print(f"\nColumnas en el DataFrame final:")
        for i, col in enumerate(df_danes.columns, 1):
            print(f"  {i}. {col}")
        
        # Mostrar resumen por origen
        print(f"\nResumen por archivo/hoja:")
        resumen_origen = df_danes.groupby(['ARCHIVO_ORIGEN', 'HOJA_ORIGEN']).size().reset_index(name='REGISTROS')
        print(resumen_origen.to_string(index=False))
        
        # Mostrar primeras filas
        print(f"\nPrimeras 5 filas del DataFrame consolidado:")
        print(df_danes.head().to_string())
        
        # PASO 6: DATOS PREPARADOS PARA EXPORTACIÓN FINAL
        print(f"\n✅ DATOS DAN PREPARADOS:")
        print(f"📋 df_danes_agrupado: {df_danes_agrupado.shape[0] if df_danes_agrupado is not None else 0} registros")
        print(f"📄 Será incluido en el archivo Excel final consolidado")
    
    else:
        print("⚠️ No se pudieron procesar datos de ningún archivo.")
        
else:
    print("⚠️ No se encontraron archivos Excel ni CSV en la carpeta DAN.")

print(f"\n🎉 PROCESAMIENTO COMPLETADO!")
print("="*60)

🔍 EXPLORANDO CARPETA DAN...
Contenido de la carpeta: C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Calculo de PC y DAN\LAM\DAN
--------------------------------------------------
1. DAN.xlsx (7.0 KB)

Total de archivos encontrados: 1

Archivos Excel encontrados: 1
- DAN.xlsx


🔍 PROCESANDO ARCHIVOS DAN (Excel y CSV)...

📂 Procesando: DAN.xlsx (Tipo: .xlsx)
   Hojas encontradas: 2 -> ['Exportar Hoja de Trabajo', 'SQL']
   ✅ Hoja 'Exportar Hoja de Trabajo': 93 filas procesadas
   ✅ Hoja 'SQL': 1 filas procesadas

✅ Columnas RUTA y CONCA creadas exitosamente
📝 Ejemplos de transformación:
   DAN62517             → RUTA: 62517 + PRODUCTO: 101304.0 = CONCA: 62517101304.0
   DAN62517             → RUTA: 62517 + PRODUCTO: 15542.0 = CONCA: 6251715542.0
   DAN62516             → RUTA: 62516 + PRODUCTO: 12583.0 = CONCA: 6251612583.0

📊 AGRUPANDO POR CONCA...
----------------------------------------
✅ Agrupación completada:
   Registros originales: 94
   Registros agrupados:

c:\Users\igcamposg\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\igcamposg\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


02. Calculo DAN con PC

In [2]:
# ==================== PROCESAMIENTO COMPLETO DE ARCHIVOS PC ====================
import pandas as pd
import os
import glob
from datetime import datetime

# Configuración de rutas
carpeta_pc = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Calculo de PC y DAN\LAM\PC'
carpeta_outputs = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Calculo de PC y DAN\LAM\Outputs'

# Crear carpeta de outputs si no existe
os.makedirs(carpeta_outputs, exist_ok=True)

# PASO 1: EXPLORACIÓN DE LA CARPETA PC
print("🔍 EXPLORANDO CARPETA PC...")
print("="*60)

if os.path.exists(carpeta_pc):
    print(f"Contenido de la carpeta: {carpeta_pc}")
    print("-" * 50)
    
    # Listar todos los archivos en la carpeta
    archivos = os.listdir(carpeta_pc)
    
    if archivos:
        for i, archivo in enumerate(archivos, 1):
            ruta_completa = os.path.join(carpeta_pc, archivo)
            tamaño = os.path.getsize(ruta_completa) / 1024  # Tamaño en KB
            print(f"{i}. {archivo} ({tamaño:.1f} KB)")
            
        print(f"\nTotal de archivos encontrados: {len(archivos)}")
        
        # Buscar archivos Excel y CSV específicamente
        archivos_excel = glob.glob(os.path.join(carpeta_pc, '*.xlsx')) + glob.glob(os.path.join(carpeta_pc, '*.xlsm')) + glob.glob(os.path.join(carpeta_pc, '*.xls'))
        archivos_csv = glob.glob(os.path.join(carpeta_pc, '*.csv'))
        
        if archivos_excel:
            print(f"\nArchivos Excel encontrados: {len(archivos_excel)}")
            for excel_file in archivos_excel:
                print(f"- {os.path.basename(excel_file)}")
        
        if archivos_csv:
            print(f"\nArchivos CSV encontrados: {len(archivos_csv)}")
            for csv_file in archivos_csv:
                print(f"- {os.path.basename(csv_file)}")
                
        if not archivos_excel and not archivos_csv:
            print("\nNo se encontraron archivos Excel ni CSV en la carpeta.")
            
    else:
        print("La carpeta está vacía.")
        
else:
    print(f"La carpeta no existe: {carpeta_pc}")

print("\n" + "="*60)

# PASO 2: PROCESAMIENTO AUTOMÁTICO DE ARCHIVOS
print("\n🔍 PROCESANDO ARCHIVOS PC (Excel y CSV)...")
print("="*60)

# Inicializar lista para almacenar todos los DataFrames
list_df_pc = []

# Buscar todos los archivos Excel y CSV en la carpeta PC
archivos_excel = glob.glob(os.path.join(carpeta_pc, '*.xlsx')) + \
                 glob.glob(os.path.join(carpeta_pc, '*.xlsm')) + \
                 glob.glob(os.path.join(carpeta_pc, '*.xls'))
archivos_csv = glob.glob(os.path.join(carpeta_pc, '*.csv'))

# Combinar todos los archivos
todos_archivos = archivos_excel + archivos_csv

if todos_archivos:
    for archivo in todos_archivos:
        nombre_archivo = os.path.basename(archivo)
        extension = os.path.splitext(archivo)[1].lower()
        print(f"\n📂 Procesando: {nombre_archivo} (Tipo: {extension})")
        
        try:
            if extension == '.csv':
                # Procesar archivo CSV
                try:
                    # Intentar leer con diferentes encodings
                    encodings = ['utf-8', 'latin-1', 'cp1252', 'iso-8859-1']
                    df_temp = None
                    
                    for encoding in encodings:
                        try:
                            df_temp = pd.read_csv(archivo, encoding=encoding)
                            print(f"   ✅ CSV leído exitosamente con encoding: {encoding}")
                            break
                        except UnicodeDecodeError:
                            continue
                    
                    if df_temp is None:
                        print(f"   ❌ No se pudo leer el CSV con ningún encoding probado")
                        continue
                    
                    if not df_temp.empty:
                        # Agregar columnas de metadatos para identificar origen
                        df_temp['ARCHIVO_ORIGEN'] = nombre_archivo
                        df_temp['HOJA_ORIGEN'] = 'CSV_DATA'
                        
                        # Estandarizar nombres de columnas
                        df_temp.columns = df_temp.columns.str.upper().str.strip()
                        
                        # Agregar a la lista
                        list_df_pc.append(df_temp)
                        print(f"   ✅ CSV procesado: {df_temp.shape[0]} filas")
                    else:
                        print(f"   ⚠️ CSV vacío, omitido")
                        
                except Exception as e:
                    print(f"   ❌ Error procesando CSV: {e}")
                    
            else:
                # Procesar archivo Excel
                excel_file = pd.ExcelFile(archivo)
                print(f"   Hojas encontradas: {len(excel_file.sheet_names)} -> {excel_file.sheet_names}")
                
                # Procesar cada hoja del archivo
                for hoja in excel_file.sheet_names:
                    try:
                        df_temp = pd.read_excel(archivo, sheet_name=hoja)
                        
                        if not df_temp.empty:
                            # Agregar columnas de metadatos para identificar origen
                            df_temp['ARCHIVO_ORIGEN'] = nombre_archivo
                            df_temp['HOJA_ORIGEN'] = hoja
                            
                            # Estandarizar nombres de columnas
                            df_temp.columns = df_temp.columns.str.upper().str.strip()
                            
                            # Agregar a la lista
                            list_df_pc.append(df_temp)
                            print(f"   ✅ Hoja '{hoja}': {df_temp.shape[0]} filas procesadas")
                        else:
                            print(f"   ⚠️ Hoja '{hoja}': vacía, omitida")
                            
                    except Exception as e:
                        print(f"   ❌ Error procesando hoja '{hoja}': {e}")
                        
        except Exception as e:
            print(f"❌ Error procesando archivo '{nombre_archivo}': {e}")

    # PASO 3: CONSOLIDACIÓN Y TRANSFORMACIÓN DE DATOS
    if list_df_pc:
        df_pc = pd.concat(list_df_pc, ignore_index=True)
        
        # Crear la columna CONCA usando LOTETERMINADO y TRIM(DE.CLAVEPRODUCTO)
        # Buscar las columnas específicas de PC (pueden tener diferentes nombres)
        columna_lote = None
        columna_producto = None
        
        # Identificar las columnas correctas
        for col in df_pc.columns:
            col_str = str(col).upper()  # Convertir a string antes de usar upper()
            if 'LOTETERMINADO' in col_str:
                columna_lote = col
            elif col_str == 'TRIM(DE.CLAVEPRODUCTO)':
                columna_producto = col
        
        # Si no encuentra la columna exacta, buscar una que contenga solo TRIM(DE.CLAVEPRODUCTO)
        if columna_producto is None:
            for col in df_pc.columns:
                col_str = str(col).upper()  # Convertir a string antes de usar upper()
                if 'TRIM(DE.CLAVEPRODUCTO)' in col_str and 'SELECT' not in col_str:
                    columna_producto = col
                    break
        
        if columna_lote is not None and columna_producto is not None:
            print(f"🔍 Columnas identificadas:")
            print(f"   - Lote: {columna_lote}")
            print(f"   - Producto: {columna_producto}")
            
            def extraer_ruta_pc(loteterminado):
                if pd.isna(loteterminado):
                    return None
                # Extraer los últimos 5 dígitos de LOTETERMINADO
                lote_str = str(loteterminado).strip()
                if len(lote_str) >= 5:
                    return lote_str[-5:]  # Últimos 5 caracteres
                else:
                    return lote_str  # Si tiene menos de 5 caracteres, devolver todo
            
            def crear_conca_pc(row):
                ruta = extraer_ruta_pc(row[columna_lote])
                claveproducto = row[columna_producto]
                
                if pd.isna(ruta) or pd.isna(claveproducto):
                    return None
                # Concatenar RUTA + TRIM(DE.CLAVEPRODUCTO)
                return f"{ruta}{claveproducto}"
            
            # Crear columna RUTA
            df_pc['RUTA'] = df_pc[columna_lote].apply(extraer_ruta_pc)
            
            # Crear columna CONCA
            df_pc['CONCA'] = df_pc.apply(crear_conca_pc, axis=1)
            
            print(f"\n✅ Columnas RUTA y CONCA creadas exitosamente")
            
            # Crear columnas de clasificación por tipo de material
            # Definir los códigos de productos por categoría
            codigos_calentadores = {45270, 45272, 45273, 45274, 49965, 49966}
            codigos_escaleras = {
                16740, 16763, 24117, 24118, 24119, 24120, 100495, 101883, 101903, 101904, 
                101905, 101906, 10264, 10435, 10438, 10445, 10495, 16741, 16742, 16764, 
                16765, 24122, 100224, 101884, 10335, 10437, 10439, 10442, 10450, 10459, 
                10496, 10497, 16743, 16744, 16746, 16747, 100225, 100226, 10337, 10338, 
                10436, 10460, 10469, 10498, 16026, 16027, 16028, 16748, 16757, 16758, 
                16759, 100227, 100228, 100229
            }
            
            def clasificar_material(claveproducto):
                if pd.isna(claveproducto):
                    return 0, 0  # (calentador, escalera)
                try:
                    codigo = int(float(str(claveproducto).strip()))
                    calentador = 1 if codigo in codigos_calentadores else 0
                    escalera = 1 if codigo in codigos_escaleras else 0
                    return calentador, escalera
                except (ValueError, TypeError):
                    return 0, 0
            
            # Aplicar clasificación
            clasificacion = df_pc[columna_producto].apply(clasificar_material)
            df_pc['Calentadores'] = [x[0] for x in clasificacion]
            df_pc['Escaleras'] = [x[1] for x in clasificacion]
            
            print(f"✅ Columnas de clasificación creadas: Calentadores y Escaleras")
            
            # Crear columna DAN+OTROS (diferencia entre CANTIDADASIGNADA y CANTIDADCARGADA)
            # Buscar las columnas de cantidad
            columna_asignada = None
            columna_cargada = None
            
            for col in df_pc.columns:
                col_str = str(col).upper()  # Convertir a string antes de usar upper()
                if 'CANTIDADASIGNADA' in col_str:
                    columna_asignada = col
                elif 'CANTIDADCARGADA' in col_str:
                    columna_cargada = col
            
            if columna_asignada is not None and columna_cargada is not None:
                print(f"🔍 Columnas de cantidad identificadas:")
                print(f"   - Cantidad Asignada: {columna_asignada}")
                print(f"   - Cantidad Cargada: {columna_cargada}")
                
                # Crear columna DAN+OTROS
                df_pc['DAN+OTROS'] = df_pc[columna_asignada] - df_pc[columna_cargada]
                
                print(f"✅ Columna DAN+OTROS creada (CANTIDADASIGNADA - CANTIDADCARGADA)")
                
                # Mostrar estadísticas de DAN+OTROS
                print(f"📊 Estadísticas de DAN+OTROS:")
                print(f"   - Total DAN+OTROS: {df_pc['DAN+OTROS'].sum():,.0f}")
                print(f"   - Promedio: {df_pc['DAN+OTROS'].mean():.2f}")
                print(f"   - Valores positivos: {(df_pc['DAN+OTROS'] > 0).sum()} registros")
                print(f"   - Valores negativos: {(df_pc['DAN+OTROS'] < 0).sum()} registros")
                print(f"   - Valores cero: {(df_pc['DAN+OTROS'] == 0).sum()} registros")
                
            else:
                print("⚠️ No se encontraron las columnas CANTIDADASIGNADA o CANTIDADCARGADA para crear DAN+OTROS")
                columnas_cantidad = [col for col in df_pc.columns if 'CANTIDAD' in str(col).upper()]
                print(f"Columnas de cantidad disponibles: {columnas_cantidad}")
            
            # Mostrar estadísticas de clasificación
            total_calentadores = df_pc['Calentadores'].sum()
            total_escaleras = df_pc['Escaleras'].sum()
            total_otros = len(df_pc) - total_calentadores - total_escaleras
            
            print(f"📊 Estadísticas de clasificación:")
            print(f"   - Calentadores: {total_calentadores} registros")
            print(f"   - Escaleras: {total_escaleras} registros")
            print(f"   - Otros productos: {total_otros} registros")
            
            # Mostrar ejemplos de clasificación
            ejemplos_clasificacion = df_pc[[columna_producto, 'Calentadores', 'Escaleras']].dropna().head(5)
            if not ejemplos_clasificacion.empty:
                print(f"\n📝 Ejemplos de clasificación:")
                for _, row in ejemplos_clasificacion.iterrows():
                    tipo = "Calentador" if row['Calentadores'] == 1 else ("Escalera" if row['Escaleras'] == 1 else "Otro")
                    print(f"   Código: {row[columna_producto]} → {tipo}")
            
            # Mostrar algunos ejemplos de la transformación CONCA
            ejemplos = df_pc[[columna_lote, 'RUTA', columna_producto, 'CONCA', 'Calentadores', 'Escaleras', 'DAN+OTROS']].dropna().head(3)
            if not ejemplos.empty:
                print("📝 Ejemplos de transformación:")
                for _, row in ejemplos.iterrows():
                    tipo = "Calentador" if row['Calentadores'] == 1 else ("Escalera" if row['Escaleras'] == 1 else "Otro")
                    print(f"   {row[columna_lote]} → RUTA: {row['RUTA']} + TRIM(CLAVEPRODUCTO): {row[columna_producto]} = CONCA: {row['CONCA']} [{tipo}]")
        else:
            print("⚠️ No se encontraron las columnas LOTETERMINADO o TRIM(DE.CLAVEPRODUCTO) para crear CONCA")
            print(f"Columnas disponibles: {list(df_pc.columns)}")
            print("📝 Buscando columnas que contengan 'TRIM' y 'CLAVEPRODUCTO':")
            for col in df_pc.columns:
                col_str = str(col).upper()
                if 'TRIM' in col_str and 'CLAVEPRODUCTO' in col_str:
                    print(f"   - {col}")
        
        # PASO 4: RESUMEN DE RESULTADOS
        print(f"\n📊 RESULTADO CONSOLIDADO:")
        print("="*60)
        print(f"Total de registros: {df_pc.shape[0]}")
        print(f"Total de columnas: {df_pc.shape[1]}")
        
        print(f"\nColumnas en el DataFrame final:")
        for i, col in enumerate(df_pc.columns, 1):
            print(f"  {i}. {col}")
        
        # Mostrar resumen por origen
        print(f"\nResumen por archivo/hoja:")
        resumen_origen = df_pc.groupby(['ARCHIVO_ORIGEN', 'HOJA_ORIGEN']).size().reset_index(name='REGISTROS')
        print(resumen_origen.to_string(index=False))
        
        # Mostrar primeras filas
        print(f"\nPrimeras 5 filas del DataFrame consolidado:")
        print(df_pc.head().to_string())
        
        # PASO 5: EXPORTACIÓN A EXCEL
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        nombre_output = f'df_pc_consolidado_{timestamp}.xlsx'
        ruta_output = os.path.join(carpeta_outputs, nombre_output)
        
        try:
            # Crear DataFrame agrupado por CONCA para la segunda hoja
            if 'CONCA' in df_pc.columns and columna_asignada is not None and columna_cargada is not None:
                df_pc_agrupado = df_pc.groupby('CONCA').agg({
                    'RUTA': 'first',
                    columna_producto: 'first',
                    columna_asignada: 'sum',
                    columna_cargada: 'sum',
                    'DAN+OTROS': 'sum'
                }).reset_index()
                
                # Renombrar columnas para el reporte
                df_pc_agrupado = df_pc_agrupado.rename(columns={
                    columna_producto: 'TRIM(DE.CLAVEPRODUCTO)',
                    columna_asignada: 'Suma de CANTIDADASIGNADA',
                    columna_cargada: 'Suma de CANTIDADCARGADA',
                    'DAN+OTROS': 'Suma de DAN+OTROS'
                })
                
                print(f"\n✅ DataFrame agrupado creado:")
                print(f"   Registros agrupados: {df_pc_agrupado.shape[0]}")
                print(f"   Columnas: {list(df_pc_agrupado.columns)}")
            
            # DATOS PREPARADOS PARA EXPORTACIÓN FINAL
            print(f"\n✅ DATOS PC PREPARADOS:")
            if 'df_pc_agrupado' in locals():
                print(f"📋 DataFrame creado:")
                print(f"   - df_pc_agrupado (Consolidado): {df_pc_agrupado.shape[0]} registros")
            print(f"📄 Será incluido en el archivo Excel final consolidado")
            
        except Exception as e:
            print(f"❌ Error al exportar archivo: {e}")
    
    else:
        print("⚠️ No se pudieron procesar datos de ningún archivo.")
        
else:
    print("⚠️ No se encontraron archivos Excel ni CSV en la carpeta PC.")

print(f"\n🎉 PROCESAMIENTO PC COMPLETADO!")
print("="*60)

🔍 EXPLORANDO CARPETA PC...
Contenido de la carpeta: C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Calculo de PC y DAN\LAM\PC
--------------------------------------------------
1. PC.xlsx (1111.4 KB)

Total de archivos encontrados: 1

Archivos Excel encontrados: 1
- PC.xlsx


🔍 PROCESANDO ARCHIVOS PC (Excel y CSV)...

📂 Procesando: PC.xlsx (Tipo: .xlsx)
   Hojas encontradas: 1 -> ['Exportar Hoja de Trabajo']
   ✅ Hoja 'Exportar Hoja de Trabajo': 21132 filas procesadas
🔍 Columnas identificadas:
   - Lote: LOTETERMINADO
   - Producto: TRIM(DE.CLAVEPRODUCTO)

✅ Columnas RUTA y CONCA creadas exitosamente
✅ Columnas de clasificación creadas: Calentadores y Escaleras
🔍 Columnas de cantidad identificadas:
   - Cantidad Asignada: CANTIDADASIGNADA
   - Cantidad Cargada: CANTIDADCARGADA
✅ Columna DAN+OTROS creada (CANTIDADASIGNADA - CANTIDADCARGADA)
📊 Estadísticas de DAN+OTROS:
   - Total DAN+OTROS: 4,234
   - Promedio: 0.20
   - Valores positivos: 152 registros
   - Valor

04. Simulador de DANES

In [3]:
# ==================SIMULADOR DE EMPAQUES Y ESPECIFICACIONES ==================
print("🔧 INICIANDO SIMULADOR DE EMPAQUES...")
print("="*60)

# --- 1. CONFIGURACIÓN ---
archivo_especificaciones = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Calculo de PC y DAN\ZQDIS.xlsx'
hoja_especificaciones = 'Especificaciones'

# --- 2. LECTURA Y CREACIÓN DEL DICCIONARIO MAESTRO ---
try:
    print("📖 Leyendo archivo de especificaciones...")
    df_especificaciones = pd.read_excel(archivo_especificaciones, sheet_name=hoja_especificaciones)
    print(f"✅ Archivo leído correctamente: {len(df_especificaciones)} registros encontrados")

    # Mapeo de columnas para estandarizar nombres
    columnas_a_renombrar = {
        'Piezas por presentación': 'Piezas',
        'Volumen (cm3)': 'Volumen',
        'Peso neto (kg)': 'Peso'
    }
    df_especificaciones = df_especificaciones.rename(columns=columnas_a_renombrar)

    # Seleccionar solo columnas necesarias
    columnas_necesarias = ['SKU', 'Presentación', 'Piezas', 'Volumen', 'Peso']
    df_final = df_especificaciones[columnas_necesarias].copy()
    
    # MANEJO DE DUPLICADOS: Ordenar por volumen descendente y eliminar duplicados
    # Estrategia: Si volumen es igual → se queda el primero, si no → se queda el de mayor volumen
    registros_antes = len(df_final)
    df_final = df_final.sort_values(['SKU', 'Presentación', 'Volumen'], ascending=[True, True, False])
    df_final = df_final.drop_duplicates(subset=['SKU', 'Presentación'], keep='first')
    registros_despues = len(df_final)
    
    duplicados_eliminados = registros_antes - registros_despues
    if duplicados_eliminados > 0:
        print(f"⚠️ Se encontraron y resolvieron {duplicados_eliminados} duplicados")
        print(f"   Estrategia: Mayor volumen (o primero si volúmenes son iguales)")
    
    # Crear diccionario optimizado
    specs_map = df_final.set_index(['SKU', 'Presentación']).to_dict('index')
    
    print(f"✅ Diccionario de especificaciones creado: {len(specs_map)} entradas únicas")
    
    # Mostrar ejemplos del diccionario
    print("\n📝 Ejemplos del diccionario de especificaciones:")
    primeros_elementos = list(specs_map.items())[:3]
    for elemento in primeros_elementos:
        print(f"   {elemento[0]} → {elemento[1]}")

except FileNotFoundError:
    print(f"❌ ERROR: No se encontró el archivo '{archivo_especificaciones}'")
    print(f"⚠️ Verifica que la ruta sea correcta y que el archivo exista.")
    specs_map = {}  # Crear diccionario vacío para evitar errores posteriores
except Exception as e:
    print(f"❌ ERROR al crear diccionario: {e}")
    import traceback
    traceback.print_exc()
    specs_map = {}  # Crear diccionario vacío para evitar errores posteriores

# --- 3. FUNCIÓN DE OPTIMIZACIÓN DE EMPAQUES ---
def calcular_empaque_completo(fila, specs_map):
    """
    Calcula la distribución óptima de empaques para una cantidad específica
    usando estrategia greedy: Masters → Inners → Piezas (SIN PALLETS)
    """
    codigo = fila['CLAVEPRODUCTO']
    cantidad_a_enviar = fila['CANTIDAD']
    
    # Obtener especificaciones por tipo de empaque (SOLO Master, Inner, Pieza)
    master_info = specs_map.get((codigo, 'Master'), {})
    inner_info = specs_map.get((codigo, 'Inner'), {})
    pieza_info = specs_map.get((codigo, 'Pieza'), {})

    # Extraer datos de cada tipo de empaque (PALLETS EXCLUIDOS)
    piezas_master = master_info.get('Piezas', 0)
    volumen_master = master_info.get('Volumen', 0)
    peso_master = master_info.get('Peso', 0)

    piezas_inner = inner_info.get('Piezas', 0)
    volumen_inner = inner_info.get('Volumen', 0)
    peso_inner = inner_info.get('Peso', 0)

    piezas_unitario = pieza_info.get('Piezas', 1)  # Default 1 para piezas
    volumen_unitario = pieza_info.get('Volumen', 0)
    peso_unitario = pieza_info.get('Peso', 0)

    # Algoritmo greedy de optimización de empaques (SIN PALLETS)
    cantidad_restante = cantidad_a_enviar
    
    # PALLETS = 0 (EXCLUIDOS DE LA ESTRATEGIA)
    cant_pallets = 0
    piezas_pallet = 0
    volumen_pallet = 0
    peso_pallet = 0
    resto_pallet = cantidad_restante  # Todo pasa directo a Masters
    
    # 1. Masters
    cant_masters = cantidad_restante // piezas_master if piezas_master > 0 else 0
    resto_master = cantidad_restante % piezas_master if piezas_master > 0 else cantidad_restante
    
    # 2. Inners
    cant_inners = resto_master // piezas_inner if piezas_inner > 0 else 0
    resto_inner = resto_master % piezas_inner if piezas_inner > 0 else resto_master
    
    # 3. Piezas sueltas
    cant_piezas = resto_inner // piezas_unitario if piezas_unitario > 0 else 0
    resto_final = resto_inner % piezas_unitario if piezas_unitario > 0 else resto_inner

    # Cálculos de volumen y peso totales (SIN PALLETS)
    volumen_total = (cant_masters * volumen_master) + (cant_inners * volumen_inner) + (cant_piezas * volumen_unitario)
    peso_total = (cant_masters * peso_master) + (cant_inners * peso_inner) + (cant_piezas * peso_unitario)

    # Check de validación
    check_warning = "⚠️ Aguas" if resto_final != 0 else "✅ OK"

    return pd.Series([
        volumen_total, peso_total, cant_pallets, cant_masters, cant_inners, cant_piezas,
        piezas_pallet, volumen_pallet, peso_pallet, resto_pallet, piezas_master,
        volumen_master, peso_master, resto_master, piezas_inner, volumen_inner,
        peso_inner, resto_inner, piezas_unitario, volumen_unitario, peso_unitario,
        resto_final, check_warning
    ])

# --- 4. DEMOSTRACIÓN CON DATOS DE EJEMPLO ---
print(f"\n🧪 PRUEBA RÁPIDA DEL SIMULADOR:")
print("-" * 40)

# Buscar un SKU real del diccionario para demostrar
skus_disponibles = [key[0] for key in specs_map.keys()]
if skus_disponibles:
    sku_ejemplo = skus_disponibles[0]
    print(f"📦 Usando SKU de ejemplo: {sku_ejemplo}")
    
    # Crear datos de prueba
    datos_prueba = pd.DataFrame({
        'CLAVEPRODUCTO': [sku_ejemplo],
        'CANTIDAD': [1250]  # Cantidad de ejemplo
    })
    
    # Ejecutar cálculo
    resultado_prueba = calcular_empaque_completo(datos_prueba.iloc[0], specs_map)
    
    print(f"🎯 Resultado para {1250} piezas (Estrategia SIN PALLETS):")
    print(f"   Pallets: {resultado_prueba[2]} (EXCLUIDOS)")
    print(f"   Masters: {resultado_prueba[3]}")
    print(f"   Inners: {resultado_prueba[4]}")
    print(f"   Piezas: {resultado_prueba[5]}")
    print(f"   Volumen total: {resultado_prueba[0]:.2f} m³")
    print(f"   Peso total: {resultado_prueba[1]:.2f} kg")
    print(f"   Estado: {resultado_prueba[22]}")

# --- 5. PROCESAMIENTO AUTOMÁTICO CON DATOS DE DAN ---
print(f"\n🚀 PROCESANDO DATOS DE DAN CON SIMULADOR DE EMPAQUES...")
print("-" * 60)

# Verificar si tenemos los datos de DAN agrupados disponibles
if 'df_danes_agrupado' in locals():
    # Usar el DataFrame de DAN agrupado
    df_pedidos = df_danes_agrupado.copy()
    print(f"✅ Usando df_danes_agrupado: {df_pedidos.shape[0]} registros")
    
    # Limpiar la columna de código de producto
    df_pedidos['CLAVEPRODUCTO'] = df_pedidos['CLAVEPRODUCTO'].astype(str).str.strip()
    
    # CONVERSIÓN A FORMATO NUMÉRICO PARA BÚSQUEDAS EN DICCIONARIOS
    print(f"\n🔢 CONVIRTIENDO DATOS A FORMATO NUMÉRICO PARA BÚSQUEDAS...")
    print("-" * 50)
    
    # Mostrar tipos originales
    print(f"📊 Tipos de datos originales:")
    print(f"   - RUTA: {df_pedidos['RUTA'].dtype}")
    print(f"   - CLAVEPRODUCTO: {df_pedidos['CLAVEPRODUCTO'].dtype}")
    print(f"   - CANTIDAD: {df_pedidos['CANTIDAD'].dtype}")
    
    # Convertir RUTA a numérico
    def convertir_a_numero_seguro(valor):
        try:
            # Remover espacios y convertir a string primero
            valor_str = str(valor).strip()
            # Intentar convertir a entero
            return int(float(valor_str))
        except (ValueError, TypeError):
            return valor  # Mantener valor original si no se puede convertir
    
    # Aplicar conversión a las columnas necesarias
    df_pedidos['RUTA'] = df_pedidos['RUTA'].apply(convertir_a_numero_seguro)
    df_pedidos['CLAVEPRODUCTO'] = df_pedidos['CLAVEPRODUCTO'].apply(convertir_a_numero_seguro)
    
    # Verificar tipos después de conversión
    print(f"\n📊 Tipos de datos después de conversión:")
    print(f"   - RUTA: {df_pedidos['RUTA'].dtype}")
    print(f"   - CLAVEPRODUCTO: {df_pedidos['CLAVEPRODUCTO'].dtype}")
    
    # Verificar que las conversiones fueron exitosas
    rutas_numericas = pd.to_numeric(df_pedidos['RUTA'], errors='coerce').notna().sum()
    productos_numericos = pd.to_numeric(df_pedidos['CLAVEPRODUCTO'], errors='coerce').notna().sum()
    total_registros = len(df_pedidos)
    
    print(f"\n✅ RESULTADOS DE CONVERSIÓN:")
    print(f"   - RUTA: {rutas_numericas}/{total_registros} registros convertidos ({(rutas_numericas/total_registros*100):.1f}%)")
    print(f"   - CLAVEPRODUCTO: {productos_numericos}/{total_registros} registros convertidos ({(productos_numericos/total_registros*100):.1f}%)")
    
    # Mostrar algunos ejemplos de datos convertidos para verificar
    print(f"\n📝 EJEMPLOS DE DATOS CONVERTIDOS:")
    ejemplos_conversion = df_pedidos[['RUTA', 'CLAVEPRODUCTO', 'CANTIDAD']].head(3)
    for _, row in ejemplos_conversion.iterrows():
        print(f"   RUTA: {row['RUTA']} (tipo: {type(row['RUTA'])}) + PRODUCTO: {row['CLAVEPRODUCTO']} (tipo: {type(row['CLAVEPRODUCTO'])}) → Cantidad: {row['CANTIDAD']}")
    
    # Verificar si los códigos convertidos existen en el diccionario specs_map
    print(f"\n🔍 VERIFICANDO EXISTENCIA EN DICCIONARIO DE ESPECIFICACIONES:")
    productos_ejemplo = df_pedidos['CLAVEPRODUCTO'].unique()[:5]
    coincidencias_encontradas = 0
    
    for producto in productos_ejemplo:
        encontrado_master = (producto, 'Master') in specs_map
        encontrado_inner = (producto, 'Inner') in specs_map
        encontrado_pieza = (producto, 'Pieza') in specs_map
        
        if any([encontrado_master, encontrado_inner, encontrado_pieza]):
            coincidencias_encontradas += 1
            print(f"   ✅ Producto {producto}: Master={encontrado_master}, Inner={encontrado_inner}, Pieza={encontrado_pieza}")
        else:
            print(f"   ❌ Producto {producto}: No encontrado en diccionario")
    
    print(f"\n📊 RESUMEN DE COINCIDENCIAS:")
    print(f"   - Productos verificados: {len(productos_ejemplo)}")
    print(f"   - Productos encontrados en diccionario: {coincidencias_encontradas}")
    print(f"   - Porcentaje de coincidencias: {(coincidencias_encontradas/len(productos_ejemplo)*100):.1f}%")
    
    if coincidencias_encontradas == 0:
        print(f"\n⚠️ ATENCIÓN: No se encontraron coincidencias en el diccionario de especificaciones")
        print(f"   Esto puede deberse a:")
        print(f"   - Códigos de producto diferentes entre los datos y el diccionario")
        print(f"   - Formato de los códigos en el archivo ZQDIS.xlsx")
        print(f"   - Verificar que la columna SKU en ZQDIS.xlsx coincida con TRIM(DE.CLAVEPRODUCTO)")
        
        # Mostrar algunos ejemplos del diccionario para comparar
        print(f"\n📝 EJEMPLOS DE CÓDIGOS EN EL DICCIONARIO:")
        productos_diccionario = list(set([key[0] for key in specs_map.keys()]))[:5]
        for producto in productos_diccionario:
            print(f"   - {producto} (tipo: {type(producto)})")
    else:
        print(f"\n✅ ¡Excelente! Se encontraron coincidencias. El simulador debería funcionar correctamente.")
    
    # Definir columnas de resultado
    columnas_resultado = [
        'Volumen m3', 'Peso Kg', 'Pallet', 'Master', 'Inner', 'Pieza',
        'Piezas Pallet', 'Volumen Pallet', 'Peso Pallet', 'R1', 'Piezas Master',
        'Volumen Master', 'Peso Master', 'R2', 'Piezas Inner', 'Volumen Inner',
        'Peso Inner', 'R3', 'Piezas Unitario', 'Volumen Unitario', 'Peso Unitario',
        'RF', 'Check'
    ]
    
    try:
        print("\n🔄 Ejecutando cálculos de empaque...")
        resultados = df_pedidos.apply(lambda fila: calcular_empaque_completo(fila, specs_map), axis=1)
        resultados.columns = columnas_resultado
        
        # Combinar datos originales con resultados
        df_final_completo = pd.concat([df_pedidos, resultados], axis=1)
        
        # VOLUMEN: Se mantiene sin conversión adicional para evitar hiperreducción
        print(f"\n🔄 CONVIRTIENDO UNIDADES DE VOLUMEN...")
        print(f"   - Volumen original (promedio): {df_final_completo['Volumen m3'].mean():.6f}")
        
        print(f"   - Volumen final (promedio): {df_final_completo['Volumen m3'].mean():.6f}")
        print(f"   ✅ No se aplica división entre 1,000,000 para evitar doble conversión")
        
        print(f"✅ Cálculos completados exitosamente")
        print(f"   - Registros procesados: {df_final_completo.shape[0]}")
        print(f"   - Columnas totales: {df_final_completo.shape[1]}")
        
        # Mostrar estadísticas de los resultados
        print(f"\n📊 ESTADÍSTICAS DE EMPAQUE:")
        print("-" * 40)
        
        # Estadísticas generales
        print(f"\n📈 TOTALES GENERALES:")
        print(f"   - Volumen total: {df_final_completo['Volumen m3'].sum():.2f} m³")
        print(f"   - Peso total: {df_final_completo['Peso Kg'].sum():.2f} kg")
        print(f"   - Pallets totales: 0 (EXCLUIDOS DE ESTRATEGIA)")
        print(f"   - Masters totales: {df_final_completo['Master'].sum():,.0f}")
        print(f"   - Inners totales: {df_final_completo['Inner'].sum():,.0f}")
        print(f"   - Piezas sueltas: {df_final_completo['Pieza'].sum():,.0f}")
        print(f"   - Cantidad total procesada: {df_final_completo['CANTIDAD'].sum():,.0f} piezas")
        
        # Identificar productos con warnings
        warnings = df_final_completo[df_final_completo['Check'].str.contains('Aguas', na=False)]
        if not warnings.empty:
            print(f"\n⚠️ PRODUCTOS CON ADVERTENCIAS: {len(warnings)} registros")
            print("   Estos productos tienen residuos no empacables:")
            for _, row in warnings.head(5).iterrows():
                print(f"     - {row['CLAVEPRODUCTO']}: {row['RF']} piezas residuales")
        else:
            print(f"\n✅ Todos los productos se empacaron correctamente")
        
        # Mostrar primeros resultados
        print(f"\n📝 PRIMEROS 5 RESULTADOS:")
        columnas_mostrar = ['RUTA', 'CLAVEPRODUCTO', 'CANTIDAD', 
                           'Volumen m3', 'Peso Kg', 'Pallet', 'Master', 'Inner', 'Pieza', 'Check']
        columnas_existentes = [col for col in columnas_mostrar if col in df_final_completo.columns]
        print(df_final_completo[columnas_existentes].head().to_string(index=False))
        
        # Crear variable global para uso posterior
        globals()['df_empaques_calculados'] = df_final_completo
        print(f"\n🔗 DataFrame 'df_empaques_calculados' creado para uso posterior")
        print(f"📄 Los resultados se incluirán automáticamente en el archivo Excel consolidado de la celda 3")
        
        # CREAR RESUMEN: SIMULADOR DANES
        print(f"\n📊 CREANDO RESUMEN 'SIMULADOR DANES'...")
        print("-" * 50)
        
        # Agrupar por RUTA y sumar Volumen m3
        df_simulador_danes = df_final_completo.groupby('RUTA').agg({
            'Volumen m3': 'sum'
        }).reset_index()
        
        # Renombrar columna para claridad
        df_simulador_danes = df_simulador_danes.rename(columns={
            'Volumen m3': 'Suma de Volumen m3'
        })
        
        # Agregar fila de total
        total_volumen = df_simulador_danes['Suma de Volumen m3'].sum()
        total_row = pd.DataFrame({
            'RUTA': ['TOTAL'],
            'Suma de Volumen m3': [total_volumen]
        })
        df_simulador_danes = pd.concat([df_simulador_danes, total_row], ignore_index=True)
        
        print(f"✅ Resumen 'Simulador DANES' creado exitosamente")
        print(f"   - Rutas únicas: {len(df_simulador_danes) - 1}")  # -1 para excluir la fila TOTAL
        print(f"   - Volumen total: {total_volumen:.6f} m³")
        
        # Mostrar el resumen completo
        print(f"\n📝 RESUMEN 'SIMULADOR DANES':")
        print(df_simulador_danes.to_string(index=False))
        
        # Crear variable global para el resumen
        globals()['df_simulador_danes'] = df_simulador_danes
        print(f"\n🔗 DataFrame 'df_simulador_danes' creado para uso posterior")
        
    except Exception as e:
        print(f"❌ Error durante el procesamiento: {e}")
        import traceback
        traceback.print_exc()
        
else:
    print("⚠️ No se encontraron datos de DAN para procesar")
    print("   Ejecuta primero las celdas anteriores para generar df_danes_agrupado")
    print("\n📋 CONFIGURACIÓN MANUAL:")
    print("   Para procesar un archivo externo, usa:")
    print("   archivo_pedidos = 'ruta_a_tu_archivo.xlsx'")
    print("   df_pedidos = pd.read_excel(archivo_pedidos)")
    print("   # Luego aplica calcular_empaque_completo()")

print(f"\n🎉 SIMULADOR DE EMPAQUES COMPLETADO!")
print("="*60)

🔧 INICIANDO SIMULADOR DE EMPAQUES...
📖 Leyendo archivo de especificaciones...
✅ Archivo leído correctamente: 91470 registros encontrados
⚠️ Se encontraron y resolvieron 1 duplicados
   Estrategia: Mayor volumen (o primero si volúmenes son iguales)
✅ Diccionario de especificaciones creado: 91469 entradas únicas

📝 Ejemplos del diccionario de especificaciones:
   (10000, 'Inner') → {'Piezas': 3, 'Volumen': 0.02078708, 'Peso': 0.00496}
   (10000, 'Pallet') → {'Piezas': 396, 'Volumen': 3.5593305, 'Peso': 0.70472}
   (10000, 'Pieza') → {'Piezas': 1, 'Volumen': 0.006928852, 'Peso': 0.001653}

🧪 PRUEBA RÁPIDA DEL SIMULADOR:
----------------------------------------
📦 Usando SKU de ejemplo: 10000
🎯 Resultado para 1250 piezas (Estrategia SIN PALLETS):
   Pallets: 0 (EXCLUIDOS)
   Masters: 0
   Inners: 416
   Piezas: 2
   Volumen total: 8.66 m³
   Peso total: 2.07 kg
   Estado: ✅ OK

🚀 PROCESANDO DATOS DE DAN CON SIMULADOR DE EMPAQUES...
-----------------------------------------------------------

05. Simulador de CARGA

In [4]:
# ==================SIMULADOR DE EMPAQUES Y ESPECIFICACIONES PARA CARGA ==================
print("🔧 INICIANDO SIMULADOR DE EMPAQUES PARA CARGA...")
print("="*60)

# --- 1. VERIFICAR DISPONIBILIDAD DE DATOS PC ---
print("🔍 VERIFICANDO DATOS DE PC AGRUPADO...")
print("-" * 40)

if 'df_pc_agrupado' in locals():
    print(f"✅ df_pc_agrupado encontrado: {df_pc_agrupado.shape[0]} registros")
    
    # Verificar columnas necesarias
    print(f"\n📋 Columnas disponibles en df_pc_agrupado:")
    for i, col in enumerate(df_pc_agrupado.columns, 1):
        print(f"  {i}. {col}")
    
    # Verificar que existan las columnas necesarias
    columnas_requeridas = ['RUTA', 'TRIM(DE.CLAVEPRODUCTO)', 'Suma de CANTIDADCARGADA']
    columnas_encontradas = []
    for col in columnas_requeridas:
        if col in df_pc_agrupado.columns:
            columnas_encontradas.append(col)
            print(f"✅ Columna encontrada: {col}")
        else:
            print(f"❌ Columna faltante: {col}")
    
    if len(columnas_encontradas) == len(columnas_requeridas):
        print(f"\n✅ Todas las columnas requeridas están disponibles")
        
        # Preparar DataFrame para el simulador (renombrar columnas para compatibilidad)
        df_pedidos_carga = df_pc_agrupado.copy()
        df_pedidos_carga = df_pedidos_carga.rename(columns={
            'TRIM(DE.CLAVEPRODUCTO)': 'CLAVEPRODUCTO',
            'Suma de CANTIDADCARGADA': 'CANTIDAD'
        })
        
        print(f"🔄 DataFrame preparado para simulador de carga:")
        print(f"   - RUTA: {df_pedidos_carga['RUTA'].dtype}")
        print(f"   - CLAVEPRODUCTO: {df_pedidos_carga['CLAVEPRODUCTO'].dtype}")
        print(f"   - CANTIDAD: {df_pedidos_carga['CANTIDAD'].dtype}")
        
        # Mostrar estadísticas de cantidad cargada
        print(f"\n📊 Estadísticas de CANTIDAD CARGADA:")
        print(f"   - Total cantidad cargada: {df_pedidos_carga['CANTIDAD'].sum():,.0f}")
        print(f"   - Cantidad promedio: {df_pedidos_carga['CANTIDAD'].mean():.2f}")
        print(f"   - Cantidad máxima: {df_pedidos_carga['CANTIDAD'].max():,.0f}")
        print(f"   - Cantidad mínima: {df_pedidos_carga['CANTIDAD'].min():,.0f}")
        
        # --- 2. CONVERSIÓN A FORMATO NUMÉRICO PARA BÚSQUEDAS ---
        print(f"\n🔢 CONVIRTIENDO DATOS A FORMATO NUMÉRICO PARA BÚSQUEDAS...")
        print("-" * 50)
        
        # Función de conversión segura (reutilizada del simulador DANES)
        def convertir_a_numero_seguro(valor):
            try:
                valor_str = str(valor).strip()
                return int(float(valor_str))
            except (ValueError, TypeError):
                return valor
        
        # Aplicar conversión
        df_pedidos_carga['RUTA'] = df_pedidos_carga['RUTA'].apply(convertir_a_numero_seguro)
        df_pedidos_carga['CLAVEPRODUCTO'] = df_pedidos_carga['CLAVEPRODUCTO'].apply(convertir_a_numero_seguro)
        
        # Verificar conversiones
        rutas_numericas = pd.to_numeric(df_pedidos_carga['RUTA'], errors='coerce').notna().sum()
        productos_numericos = pd.to_numeric(df_pedidos_carga['CLAVEPRODUCTO'], errors='coerce').notna().sum()
        total_registros = len(df_pedidos_carga)
        
        print(f"✅ RESULTADOS DE CONVERSIÓN:")
        print(f"   - RUTA: {rutas_numericas}/{total_registros} registros convertidos ({(rutas_numericas/total_registros*100):.1f}%)")
        print(f"   - CLAVEPRODUCTO: {productos_numericos}/{total_registros} registros convertidos ({(productos_numericos/total_registros*100):.1f}%)")
        
        # --- 3. VERIFICAR EXISTENCIA EN DICCIONARIO DE ESPECIFICACIONES ---
        print(f"\n🔍 VERIFICANDO EXISTENCIA EN DICCIONARIO DE ESPECIFICACIONES:")
        
        # Verificar que existe el diccionario specs_map (del simulador DANES)
        if 'specs_map' in globals():
            productos_ejemplo = df_pedidos_carga['CLAVEPRODUCTO'].unique()[:5]
            coincidencias_encontradas = 0
            
            for producto in productos_ejemplo:
                encontrado_master = (producto, 'Master') in specs_map
                encontrado_inner = (producto, 'Inner') in specs_map
                encontrado_pieza = (producto, 'Pieza') in specs_map
                
                if any([encontrado_master, encontrado_inner, encontrado_pieza]):
                    coincidencias_encontradas += 1
                    print(f"   ✅ Producto {producto}: Master={encontrado_master}, Inner={encontrado_inner}, Pieza={encontrado_pieza}")
                else:
                    print(f"   ❌ Producto {producto}: No encontrado en diccionario")
            
            print(f"\n📊 RESUMEN DE COINCIDENCIAS PARA CARGA:")
            print(f"   - Productos verificados: {len(productos_ejemplo)}")
            print(f"   - Productos encontrados en diccionario: {coincidencias_encontradas}")
            print(f"   - Porcentaje de coincidencias: {(coincidencias_encontradas/len(productos_ejemplo)*100):.1f}%")
            
            # --- 4. EJECUTAR SIMULADOR DE CARGA ---
            if coincidencias_encontradas > 0:
                print(f"\n🚀 EJECUTANDO SIMULADOR DE EMPAQUES PARA CARGA...")
                print("-" * 60)
                
                # Definir columnas de resultado (las mismas del simulador DANES)
                columnas_resultado = [
                    'Volumen m3', 'Peso Kg', 'Pallet', 'Master', 'Inner', 'Pieza',
                    'Piezas Pallet', 'Volumen Pallet', 'Peso Pallet', 'R1', 'Piezas Master',
                    'Volumen Master', 'Peso Master', 'R2', 'Piezas Inner', 'Volumen Inner',
                    'Peso Inner', 'R3', 'Piezas Unitario', 'Volumen Unitario', 'Peso Unitario',
                    'RF', 'Check'
                ]
                
                try:
                    print("🔄 Ejecutando cálculos de empaque para CARGA...")
                    resultados_carga = df_pedidos_carga.apply(lambda fila: calcular_empaque_completo(fila, specs_map), axis=1)
                    resultados_carga.columns = columnas_resultado
                    
                    # Combinar datos originales con resultados
                    df_final_completo_carga = pd.concat([df_pedidos_carga, resultados_carga], axis=1)
                    
                    # VOLUMEN: Se mantiene sin conversión adicional para evitar hiperreducción
                    print(f"\n🔄 CONVIRTIENDO UNIDADES DE VOLUMEN...")
                    print(f"   - Volumen original (promedio): {df_final_completo_carga['Volumen m3'].mean():.6f}")
                    
                    print(f"   - Volumen final (promedio): {df_final_completo_carga['Volumen m3'].mean():.6f}")
                    print(f"   ✅ No se aplica división entre 1,000,000 para evitar doble conversión")
                    
                    print(f"✅ Cálculos completados exitosamente")
                    print(f"   - Registros procesados: {df_final_completo_carga.shape[0]}")
                    print(f"   - Columnas totales: {df_final_completo_carga.shape[1]}")
                    
                    # --- 5. ESTADÍSTICAS DE EMPAQUE PARA CARGA ---
                    print(f"\n📊 ESTADÍSTICAS DE EMPAQUE PARA CARGA:")
                    print("-" * 40)
                    
                    print(f"\n📈 TOTALES GENERALES:")
                    print(f"   - Volumen total: {df_final_completo_carga['Volumen m3'].sum():.2f} m³")
                    print(f"   - Peso total: {df_final_completo_carga['Peso Kg'].sum():.2f} kg")
                    print(f"   - Pallets totales: 0 (EXCLUIDOS DE ESTRATEGIA)")
                    print(f"   - Masters totales: {df_final_completo_carga['Master'].sum():,.0f}")
                    print(f"   - Inners totales: {df_final_completo_carga['Inner'].sum():,.0f}")
                    print(f"   - Piezas sueltas: {df_final_completo_carga['Pieza'].sum():,.0f}")
                    print(f"   - Cantidad total procesada: {df_final_completo_carga['CANTIDAD'].sum():,.0f} piezas")
                    
                    # Identificar productos con warnings
                    warnings_carga = df_final_completo_carga[df_final_completo_carga['Check'].str.contains('Aguas', na=False)]
                    if not warnings_carga.empty:
                        print(f"\n⚠️ PRODUCTOS CON ADVERTENCIAS: {len(warnings_carga)} registros")
                        print("   Estos productos tienen residuos no empacables:")
                        for _, row in warnings_carga.head(5).iterrows():
                            print(f"     - {row['CLAVEPRODUCTO']}: {row['RF']} piezas residuales")
                    else:
                        print(f"\n✅ Todos los productos se empacaron correctamente")
                    
                    # Mostrar primeros resultados
                    print(f"\n📝 PRIMEROS 5 RESULTADOS DE CARGA:")
                    columnas_mostrar = ['RUTA', 'CLAVEPRODUCTO', 'CANTIDAD', 
                                       'Volumen m3', 'Peso Kg', 'Pallet', 'Master', 'Inner', 'Pieza', 'Check']
                    columnas_existentes = [col for col in columnas_mostrar if col in df_final_completo_carga.columns]
                    print(df_final_completo_carga[columnas_existentes].head().to_string(index=False))
                    
                    # Crear variable global para uso posterior
                    globals()['df_empaques_calculados_carga'] = df_final_completo_carga
                    print(f"\n🔗 DataFrame 'df_empaques_calculados_carga' creado para uso posterior")
                    
                    # --- 6. CREAR RESUMEN: SIMULADOR CARGA ---
                    print(f"\n📊 CREANDO RESUMEN 'SIMULADOR CARGA'...")
                    print("-" * 50)
                    
                    # Agrupar por RUTA y sumar Volumen m3
                    df_simulador_carga = df_final_completo_carga.groupby('RUTA').agg({
                        'Volumen m3': 'sum'
                    }).reset_index()
                    
                    # Renombrar columna para claridad
                    df_simulador_carga = df_simulador_carga.rename(columns={
                        'Volumen m3': 'Suma de Volumen m3'
                    })
                    
                    # Agregar fila de total
                    total_volumen_carga = df_simulador_carga['Suma de Volumen m3'].sum()
                    total_row_carga = pd.DataFrame({
                        'RUTA': ['TOTAL'],
                        'Suma de Volumen m3': [total_volumen_carga]
                    })
                    df_simulador_carga = pd.concat([df_simulador_carga, total_row_carga], ignore_index=True)
                    
                    print(f"✅ Resumen 'Simulador CARGA' creado exitosamente")
                    print(f"   - Rutas únicas: {len(df_simulador_carga) - 1}")  # -1 para excluir la fila TOTAL
                    print(f"   - Volumen total: {total_volumen_carga:.6f} m³")
                    
                    # Mostrar el resumen completo
                    print(f"\n📝 RESUMEN 'SIMULADOR CARGA':")
                    print(df_simulador_carga.to_string(index=False))
                    
                    # Crear variable global para el resumen
                    globals()['df_simulador_carga'] = df_simulador_carga
                    print(f"\n🔗 DataFrame 'df_simulador_carga' creado para uso posterior")
                    print(f"📄 Los resultados se incluirán automáticamente en el archivo Excel consolidado")
                    
                except Exception as e:
                    print(f"❌ Error durante el procesamiento de CARGA: {e}")
                    import traceback
                    traceback.print_exc()
            else:
                print(f"\n❌ No se pueden procesar datos de CARGA:")
                print(f"   No se encontraron productos coincidentes en el diccionario de especificaciones")
        else:
            print(f"\n❌ No se encontró el diccionario de especificaciones 'specs_map'")
            print(f"   Ejecuta primero la celda 4 (Simulador de DANES) para crear el diccionario")
    else:
        print(f"\n❌ Faltan columnas requeridas en df_pc_agrupado")
        print(f"   Columnas faltantes: {set(columnas_requeridas) - set(columnas_encontradas)}")
else:
    print("❌ No se encontró df_pc_agrupado")
    print("   Ejecuta primero la celda 2 (Calculo DAN con PC) para generar df_pc_agrupado")

print(f"\n🎉 SIMULADOR DE CARGA COMPLETADO!")
print("="*60)

🔧 INICIANDO SIMULADOR DE EMPAQUES PARA CARGA...
🔍 VERIFICANDO DATOS DE PC AGRUPADO...
----------------------------------------
✅ df_pc_agrupado encontrado: 20757 registros

📋 Columnas disponibles en df_pc_agrupado:
  1. CONCA
  2. RUTA
  3. TRIM(DE.CLAVEPRODUCTO)
  4. Suma de CANTIDADASIGNADA
  5. Suma de CANTIDADCARGADA
  6. Suma de DAN+OTROS
✅ Columna encontrada: RUTA
✅ Columna encontrada: TRIM(DE.CLAVEPRODUCTO)
✅ Columna encontrada: Suma de CANTIDADCARGADA

✅ Todas las columnas requeridas están disponibles
🔄 DataFrame preparado para simulador de carga:
   - RUTA: object
   - CLAVEPRODUCTO: int64
   - CANTIDAD: int64

📊 Estadísticas de CANTIDAD CARGADA:
   - Total cantidad cargada: 4,920,410
   - Cantidad promedio: 237.05
   - Cantidad máxima: 32,100
   - Cantidad mínima: 0

🔢 CONVIRTIENDO DATOS A FORMATO NUMÉRICO PARA BÚSQUEDAS...
--------------------------------------------------
✅ RESULTADOS DE CONVERSIÓN:
   - RUTA: 20757/20757 registros convertidos (100.0%)
   - CLAVEPRODUCTO: 2

06. Exportación Excel Consolidado

In [5]:
# ==================== EXPORTACIÓN EXCEL CONSOLIDADO ====================
print("📄 EXPORTANDO ARCHIVO EXCEL CONSOLIDADO...")
print("="*60)

# Verificar que existan todos los DataFrames necesarios
print(f"\n🔍 VERIFICANDO DATAFRAMES DISPONIBLES:")

# DataFrames básicos (de celdas 1, 2)
dataframes_basicos = {
    'df_danes_agrupado': 'Calculo DAN',
    'df_pc_agrupado': 'Calculo PC'
}

# DataFrames del simulador DANES (de celda 4)
dataframes_simulador_danes = {
    'df_empaques_calculados': 'Simulacion Empaques DANES',
    'df_simulador_danes': 'Simulador DANES'
}

# DataFrames del simulador CARGA (de celda 5)
dataframes_simulador_carga = {
    'df_empaques_calculados_carga': 'Simulacion Empaques CARGA',
    'df_simulador_carga': 'Simulador CARGA'
}

# Verificar DataFrames básicos
disponibles_basicos = {}
for var_name, hoja_name in dataframes_basicos.items():
    if var_name in locals() and locals()[var_name] is not None:
        disponibles_basicos[var_name] = hoja_name
        print(f"   ✅ {hoja_name}: {locals()[var_name].shape[0]} registros")
    else:
        print(f"   ❌ {hoja_name}: No encontrado en locals()")

# Verificar DataFrames del simulador DANES
disponibles_simulador_danes = {}
for var_name, hoja_name in dataframes_simulador_danes.items():
    if var_name in globals() and globals()[var_name] is not None:
        disponibles_simulador_danes[var_name] = hoja_name
        df_temp = globals()[var_name]
        if var_name == 'df_simulador_danes':
            print(f"   ✅ {hoja_name}: {df_temp.shape[0]} filas x {df_temp.shape[1]} columnas (tabla dinámica)")
        else:
            print(f"   ✅ {hoja_name}: {df_temp.shape[0]} registros")
    else:
        print(f"   ❌ {hoja_name}: No encontrado en globals()")

# Verificar DataFrames del simulador CARGA
disponibles_simulador_carga = {}
for var_name, hoja_name in dataframes_simulador_carga.items():
    if var_name in globals() and globals()[var_name] is not None:
        disponibles_simulador_carga[var_name] = hoja_name
        df_temp = globals()[var_name]
        if var_name == 'df_simulador_carga':
            print(f"   ✅ {hoja_name}: {df_temp.shape[0]} filas x {df_temp.shape[1]} columnas (tabla dinámica)")
        else:
            print(f"   ✅ {hoja_name}: {df_temp.shape[0]} registros")
    else:
        print(f"   ❌ {hoja_name}: No encontrado en globals()")

# Continuar solo si hay DataFrames disponibles
total_disponibles = len(disponibles_basicos) + len(disponibles_simulador_danes) + len(disponibles_simulador_carga)

if total_disponibles > 0:
    # EXPORTACIÓN CONSOLIDADA
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    nombre_output = f'Reporte_Consolidado_DAN_PC_CARGA_{timestamp}.xlsx'
    ruta_output = os.path.join(carpeta_outputs, nombre_output)
    
    print(f"\n📊 CREANDO ARCHIVO EXCEL CONSOLIDADO:")
    print(f"   📁 Nombre: {nombre_output}")
    print(f"   📂 Ubicación: {carpeta_outputs}")
    
    try:
        with pd.ExcelWriter(ruta_output, engine='openpyxl') as writer:
            hojas_creadas = 0
            
            # Exportar DataFrames básicos
            for var_name, hoja_name in disponibles_basicos.items():
                try:
                    df = locals()[var_name]
                    df.to_excel(writer, sheet_name=hoja_name, index=False)
                    hojas_creadas += 1
                    print(f"   ✅ {hoja_name}: Exportado exitosamente")
                except Exception as e:
                    print(f"   ❌ Error exportando {hoja_name}: {e}")
            
            # Exportar DataFrames del simulador DANES
            for var_name, hoja_name in disponibles_simulador_danes.items():
                try:
                    df = globals()[var_name]
                    # Para tabla dinámica, incluir índice
                    incluir_indice = (var_name == 'df_simulador_danes')
                    df.to_excel(writer, sheet_name=hoja_name, index=incluir_indice)
                    hojas_creadas += 1
                    print(f"   ✅ {hoja_name}: Exportado exitosamente")
                except Exception as e:
                    print(f"   ❌ Error exportando {hoja_name}: {e}")
            
            # Exportar DataFrames del simulador CARGA
            for var_name, hoja_name in disponibles_simulador_carga.items():
                try:
                    df = globals()[var_name]
                    # Para tabla dinámica, incluir índice
                    incluir_indice = (var_name == 'df_simulador_carga')
                    df.to_excel(writer, sheet_name=hoja_name, index=incluir_indice)
                    hojas_creadas += 1
                    print(f"   ✅ {hoja_name}: Exportado exitosamente")
                except Exception as e:
                    print(f"   ❌ Error exportando {hoja_name}: {e}")
        
        # Mensaje de éxito
        print(f"\n💾 ARCHIVO CONSOLIDADO EXPORTADO EXITOSAMENTE!")
        print(f"📁 Ubicación: {ruta_output}")
        print(f"📄 Nombre: {nombre_output}")
        print(f"📋 Hojas creadas: {hojas_creadas}")
        
        # Resumen detallado de hojas
        print(f"\n📊 RESUMEN DE HOJAS EXPORTADAS:")
        todos_disponibles = {**disponibles_basicos, **disponibles_simulador_danes, **disponibles_simulador_carga}
        
        for var_name, hoja_name in todos_disponibles.items():
            if var_name in locals():
                df = locals()[var_name]
            else:
                df = globals()[var_name]
            
            if var_name in ['df_simulador_danes', 'df_simulador_carga']:
                print(f"   - {hoja_name}: {df.shape[0]} filas x {df.shape[1]} columnas (tabla dinámica)")
            else:
                print(f"   - {hoja_name}: {df.shape[0]} registros")
        
        print(f"\n🎯 ORDEN DE EJECUCIÓN RECOMENDADO:")
        print(f"   1. Celda 1: Calculo DAN con Reporte DANES")
        print(f"   2. Celda 2: Calculo DAN con PC") 
        print(f"   3. Celda 4: Simulador de DANES")
        print(f"   4. Celda 5: Simulador de CARGA")
        print(f"   5. Esta celda: Exportación Excel Consolidado")
        
    except Exception as e:
        print(f"❌ ERROR AL EXPORTAR ARCHIVO CONSOLIDADO:")
        print(f"   {e}")
        import traceback
        traceback.print_exc()
        
else:
    print(f"\n❌ NO SE PUEDEN EXPORTAR DATOS:")
    print(f"   No se encontraron DataFrames disponibles para exportar")
    print(f"   Ejecuta las celdas anteriores en orden para generar los datos")
    
    print(f"\n🔧 SOLUCIÓN:")
    print(f"   1. Ejecuta las celdas 1, 2, 4 y 5 en orden")
    print(f"   2. Luego ejecuta esta celda de exportación")

print(f"\n🎉 EXPORTACIÓN COMPLETADA!")
print("="*60)

📄 EXPORTANDO ARCHIVO EXCEL CONSOLIDADO...

🔍 VERIFICANDO DATAFRAMES DISPONIBLES:
   ✅ Calculo DAN: 93 registros
   ✅ Calculo PC: 20757 registros
   ✅ Simulacion Empaques DANES: 93 registros
   ✅ Simulador DANES: 19 filas x 2 columnas (tabla dinámica)
   ✅ Simulacion Empaques CARGA: 20757 registros
   ✅ Simulador CARGA: 123 filas x 2 columnas (tabla dinámica)

📊 CREANDO ARCHIVO EXCEL CONSOLIDADO:
   📁 Nombre: Reporte_Consolidado_DAN_PC_CARGA_20260407_100434.xlsx
   📂 Ubicación: C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Calculo de PC y DAN\LAM\Outputs
   ✅ Calculo DAN: Exportado exitosamente
   ✅ Calculo PC: Exportado exitosamente
   ✅ Simulacion Empaques DANES: Exportado exitosamente
   ✅ Simulador DANES: Exportado exitosamente
   ✅ Simulacion Empaques CARGA: Exportado exitosamente
   ✅ Simulador CARGA: Exportado exitosamente

💾 ARCHIVO CONSOLIDADO EXPORTADO EXITOSAMENTE!
📁 Ubicación: C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Calcu